#Prompt Management in Agentic AI

##Notes covering: versioning, A/B testing, AWS Bedrock, caching, and DSPy

##Versioning Prompts: In Code vs Managed Resources

**In Code** (hardcoded string in .py file)

* Version history comes free via Git
* Simple, good for prototypes/solo projects
* Any change needs a full redeploy
* No easy rollback or non-engineer access

**As Managed Resources** (stored outside code — DB, config, prompt tool)

* Update live, no redeploy needed
* Built-in versioning + rollback (v1, v2, v3...)
* Enables A/B testing and non-engineer editing
* Needs extra infra; risk of prompt/code drift

##A/B Testing Prompt Variants

Running two+ prompt versions side by side to see which performs better.

**Workflow**

* Define a metric first (accuracy, hallucination rate, cost, latency)
* Split traffic between Prompt A and Prompt B
* Log which version handled each request
* Score outputs — human eval or LLM-as-judge
* Roll out the winner; keep the loser archived

**Offline vs Online**

* Offline = test on fixed eval set first (safe, cheap)
* Online = test on live traffic after offline looks good

**Common pitfalls:** too few test cases, changing multiple variables at once, biased judge model

##AWS Bedrock Prompt Management

AWS's managed system for storing, versioning, testing prompts outside app code.

* **Prompt Builder** — console UI to create/edit/test prompts
* **Variables** — `{{variable}}` placeholders filled at runtime
* **Variants** — up to 3 prompt versions compared side-by-side (built-in A/B testing)
* **Versioning** — DRAFT (editable) vs saved versions (frozen, rollback-able)
* **Prompt Optimization** — auto-rewrites prompts for better accuracy
* **Flows** — chain prompts into multi-step pipelines (like LangGraph nodes)
* **Aliases** — app points to a stable alias; swap versions behind it with zero downtime

Can also be managed as Infrastructure-as-Code (CloudFormation/SAM).

##Prompt Caching

Stores the processed version of stable prompt text so repeat calls skip reprocessing it.

**Anthropic — `cache_control` (explicit)**

* Developer marks cacheable blocks manually
* Prefix cache — static content first, dynamic user input last
* Write cost: 1.25x (5 min TTL) or 2.0x (1 hr TTL) input price
* Read cost: 0.10x input price — 90% discount
* Min 1,024 tokens to be cacheable; exact match required

**OpenAI — automatic caching (implicit)**

* No code change needed, kicks in above 1,024 tokens
* Free writes
* Read discount: 50% off

**When it pays off:** long stable system prompts, high call volume, multi-turn agents

**When it doesn't:** short prompts, low-frequency calls, prefix that changes every request

##DSPy — Programmatic Prompt Optimization

Framework that lets the model optimize its own prompts, instead of manual tuning.

* **Signature** — defines input -> output shape of the task
* **Module** — defines reasoning strategy (chain-of-thought, ReAct, etc.)
* **Metric** — scoring function to measure output quality
* **Optimizer** — searches for best instructions + few-shot examples (e.g. MIPROv2, BootstrapFewShot)

Optimizes prompt text only — never touches model weights.

**Good fit:** structured tasks with an eval set (classification, extraction, multi-hop QA)

**Poor fit:** one-shot creative tasks, overall pipeline orchestration (use LangGraph for that)